In [2]:
print('Hello, World!')

Hello, World!


In [3]:
from dotenv import load_dotenv
from openai import OpenAI

In [4]:
load_dotenv()

True

In [5]:
openai_client = OpenAI()

In [6]:
def llm(prompt) :
    response = openai_client.responses.create(
        model='gpt-3.5-turbo',
        input=prompt
    )
    return response.output_text

In [7]:
llm('can I still join the course?')

"I'm not sure which course you are referring to. Can you provide more information or context so I can better assist you?"

In [8]:
question = 'I just disccovered the course, can I still join?'
answer = llm(question)
print(answer)

It's great to hear that you're interested in joining the course! The ability to join would depend on the course provider's policies regarding late enrollment. I recommend reaching out to the course provider or the instructor to inquire about any possibilities for joining the course after it has already started. They will be able to provide you with more information and guidance on how you can proceed.


In [9]:
context = '''
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

edit on GitHub
#Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

edit on GitHub
#What is the video/zoom link to the stream for the “Office Hours” or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs.

Students participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the announcements channel on Telegram &amp; Slack before it begins. You can also watch live on the DataTalksClub YouTube Channel.

Don’t post questions in chat as they may be missed if the room is very active.


'''


In [10]:
prompt = f'''
Your task is to answer the question based on the following context. If you don't know the answer, say you don't know.
Question: {question}
Context: {context}
'''
print(prompt)


Your task is to answer the question based on the following context. If you don't know the answer, say you don't know.
Question: I just disccovered the course, can I still join?
Context: 
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

edit on GitHub
#Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

edit on GitHub
#What is the video/zoom link to the stream for the “Office Hours” or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs.

Students participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live

In [11]:
answer = llm(prompt)
print(answer)

If you have just discovered the course, you can still join. However, if you want to receive a certificate, you need to submit your project while they are still accepting submissions.


In [14]:
import requests
docs_url = 'https://datatalks.club/faq/json/courses.json'
response = requests.get(docs_url)
courses_raw = response.json()
courses_raw

[{'course': 'machine-learning-zoomcamp',
  'course_name': 'ML Zoomcamp',
  'path': '/json/machine-learning-zoomcamp.json',
  'questions_count': 472},
 {'course': 'llm-zoomcamp',
  'course_name': 'LLM Zoomcamp',
  'path': '/json/llm-zoomcamp.json',
  'questions_count': 79},
 {'course': 'data-engineering-zoomcamp',
  'course_name': 'Data Engineering Zoomcamp',
  'path': '/json/data-engineering-zoomcamp.json',
  'questions_count': 402},
 {'course': 'mlops-zoomcamp',
  'course_name': 'MLOps Zoomcamp',
  'path': '/json/mlops-zoomcamp.json',
  'questions_count': 255}]

In [17]:
documents = []
url_prefix = 'https://datatalks.club/faq'
for course in courses_raw :
    course_url  = f'{url_prefix}{course['path']}'
    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()
    documents.extend(course_data)

len(documents)

1208

In [21]:
from minsearch import Index

index = Index(
    text_fields=['question', 'section','answer'],
    keyword_fields=['course']
)
index.fit(documents)

In [37]:
def search(question,course='llm-zoomcamp') :
    boost_dict = {'question': 2.0, 'section': 0.5}
    filter_dict = {'course': course}
    return index.search(question, boost_dict=boost_dict, filter_dict=filter_dict, num_results=5)

In [38]:
search(question)

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '9f689c185f',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I missed the first homework - can I still get a certificate?',
  'answer': 'Yes, you need to pass the Capstone project to get the certificate. Homework is not mandatory, though it is recommended for reinforcing concepts, and the points awarded count towards your rank on the leaderboard.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learn

In [35]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

In [36]:
USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

In [40]:
def build_context(search_results) :
    lines = []
    for doc in search_results :
        lines.append(doc['section'])
        lines.append('Q: ' + doc['question'])
        lines.append('A: ' + doc['answer'])
        lines.append('')
    return '\n'.join(lines).strip()


In [41]:
context = build_context(search(question))
print(context)

General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: I missed the first homework - can I still get a certificate?
A: Yes, you need to pass the Capstone project to get the certificate. Homework is not mandatory, though it is recommended for reinforcing concepts, and the points awarded count towards your rank on the leaderboard.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced m

In [44]:
def build_prompt(question, search_results) :
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(question=question, context=context)
    return prompt.strip()

prompt = build_prompt(question, search(question))
print(prompt)

Question:
I just disccovered the course, can I still join?

Context:
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: I missed the first homework - can I still get a certificate?
A: Yes, you need to pass the Capstone project to get the certificate. Homework is not mandatory, though it is recommended for reinforcing concepts, and the points awarded count towards your rank on the leaderboard.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Relate

In [48]:
response = openai_client.responses.create(
        model='gpt-3.5-turbo',
        input=prompt
    )

In [49]:
response.output_text

'Yes, you can still join the course. Just remember that if you want to receive a certificate, you should submit your project while submissions are still open.'

In [50]:
print(response.model_dump_json(indent=2))

{
  "id": "resp_091ef22c11d16491006a287ee8ab708191aee913afb66fb83d",
  "created_at": 1781038824.0,
  "error": null,
  "incomplete_details": null,
  "instructions": null,
  "metadata": {},
  "model": "gpt-3.5-turbo-0125",
  "object": "response",
  "output": [
    {
      "id": "msg_091ef22c11d16491006a287eea41f8819181f318e88d95da21",
      "content": [
        {
          "annotations": [],
          "text": "Yes, you can still join the course. Just remember that if you want to receive a certificate, you should submit your project while submissions are still open.",
          "type": "output_text",
          "logprobs": []
        }
      ],
      "role": "assistant",
      "status": "completed",
      "type": "message",
      "phase": null
    }
  ],
  "parallel_tool_calls": true,
  "temperature": 1.0,
  "tool_choice": "auto",
  "tools": [],
  "top_p": 1.0,
  "background": false,
  "completed_at": 1781038826.0,
  "conversation": null,
  "max_output_tokens": null,
  "max_tool_calls": nu

In [51]:
response.usage

ResponseUsage(input_tokens=341, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=32, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=373)

In [53]:
input_price = 0.75/1000000
output_price = 4.5/1000000

cost = (response.usage.input_tokens * input_price) + (response.usage.output_tokens * output_price)

cost

0.00039975000000000004

In [59]:
def llm(instructions, user_prompt,model='gpt-5.4-mini'):
    message_history = [
        {'role': 'developer', 'content': instructions},
        {'role': 'user', 'content': user_prompt}
    ]

    response = openai_client.responses.create (
            model=model,
            input=message_history
    )

    return response.output_text


In [60]:
def rag(question, model = 'gpt-5.4-mini') :
    search_results = search(question)
    prompt = build_prompt(question, search_results)
    answer = llm(INSTRUCTIONS,prompt,)
    return answer   

In [61]:
answer = rag(question)
print(answer)

Yes, you can still join.

If you want to receive a certificate, make sure to submit your project while submissions are still open.
